<h1 align="center"><b> Predicting Diabetes with Exploratory Data Analysis and Machine Learning</b></h1>



# Diabetes is one of the most common chronic diseases affecting millions of people worldwide. Early detection is important to prevent serious health complications. In this project, we perform Exploratory Data Analysis (EDA) to understand patterns and relationships in the dataset and then apply Machine Learning models to predict whether a person is likely to have diabetes based on various health-related features. The aim is to support early diagnosis and data-driven healthcare decisions.

# 1. Import Libraries

In [56]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'colab'  # For Google Colab
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns

# 2. Load the Dataset

In [57]:
df = pd.read_csv("/diabetes.csv")

# 3. Initial Data Inspection

In [58]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [60]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

In [61]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [62]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.00
Glucose,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.00
BloodPressure,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.00
SkinThickness,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.00
Insulin,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.00
BMI,768.0,31.992578,7.884160,0.000,27.30000,32.0000,36.60000,67.10
DiabetesPedigreeFunction,768.0,0.471876,0.331329,0.078,0.24375,0.3725,0.62625,2.42
Age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.00
Outcome,768.0,0.348958,0.476951,0.000,0.00000,0.0000,1.00000,1.00


# 4. Data Cleaning

# 4.1 Check Missing Values

In [63]:
df.isnull().sum()

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


# 4.2 Remove Duplicates

In [64]:
df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

# 4.3 Handle Invalid Zeros (Medical Context)

In [65]:
cols_with_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in cols_with_zero:
    df[col] = df[col].replace(0, np.nan)
    df[col].fillna(df[col].median(), inplace=True)
print(" Zeros handled.")

 Zeros handled.


# 4.4 Cap Outliers

In [66]:
def cap_outliers(df, col):
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    df[col] = df[col].clip(lower, upper)
    return df

for col in df.select_dtypes(include=np.number).columns:
    if col != 'Outcome':
        df = cap_outliers(df, col)
print(" Outliers capped.")

 Outliers capped.


# 5. Exploratory Data Analysis

# 5.1 Outcome Distribution

In [67]:
fig = px.pie(df, names='Outcome', title=' Outcome Distribution (0=No, 1=Yes)',
             color='Outcome', color_discrete_map={0:'lightblue',1:'coral'})
fig.show()

# 5.2 Glucose by Outcome

In [68]:
fig = px.histogram(df, x='Glucose', color='Outcome', barmode='overlay',
                   title=' Glucose Distribution by Outcome', opacity=0.6, marginal='box')
fig.show()

# 5.3 BMI by Outcome

In [69]:
fig = px.box(df, x='Outcome', y='BMI', color='Outcome', title=' BMI by Outcome')
fig.show()

# 5.4 Age Distribution by Outcome

In [70]:
fig = px.histogram(df, x='Age', color='Outcome', barmode='overlay',
                   title=' Age Distribution by Outcome', opacity=0.6)
fig.show()

# 5.5 Correlation Heatmap

In [71]:
corr = df.corr()
fig = px.imshow(corr, text_auto=True, title=' Correlation Heatmap',
                color_continuous_scale='RdBu', zmin=-1, zmax=1)
fig.show()

# 5.6 Scatter Matrix (Selected Features)

In [72]:
fig = px.scatter_matrix(df, dimensions=['Glucose','BMI','Age','Insulin'],
                        color='Outcome', title=' Scatter Matrix')
fig.update_traces(diagonal_visible=False)
fig.show()

# 5.7 Insulin Violin Plot

In [73]:
fig = px.violin(df, x='Outcome', y='Insulin', box=True, title=' Insulin by Outcome')
fig.show()

# 5.8 3D Scatter (Glucose, BMI, Age)

In [74]:
fig = px.scatter_3d(df, x='Glucose', y='BMI', z='Age', color='Outcome',
                    title=' 3D View: Glucose, BMI, Age')
fig.show()

# 5.9 Pregnancies by Outcome

In [75]:
fig = px.box(df, x='Outcome', y='Pregnancies', color='Outcome', title=' Pregnancies by Outcome')
fig.show()

# 5.10 Density Contour (Glucose vs BMI)

In [76]:
fig = px.density_contour(df, x='Glucose', y='BMI', color='Outcome',
                         title='Glucose vs BMI Density')
fig.show()

# 6. Feature Engineering

In [77]:
# Interaction feature
df['Glucose_BMI'] = df['Glucose'] * df['BMI']

In [78]:
# Age groups
bins = [20,30,40,50,60,100]
labels = ['20-30','30-40','40-50','50-60','60+']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)
df = pd.get_dummies(df, columns=['AgeGroup'], drop_first=True)

In [79]:
print("✅ New features added. Shape:", df.shape)

✅ New features added. Shape: (768, 14)


# 7. Train-Test Split & Scaling

In [80]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

In [81]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")

Train: 614 samples, Test: 154 samples


In [82]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features scaled.")

Features scaled.


# 8. Train Multiple Models

In [83]:
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

In [84]:
predictions = {}
probabilities = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    predictions[name] = model.predict(X_test_scaled)
    probabilities[name] = model.predict_proba(X_test_scaled)[:,1]
    print(f"{name} trained.")

Logistic Regression trained.
Decision Tree trained.
Random Forest trained.
SVM trained.


# 9. Model Evaluation

# 9.1 Define Evaluation Function

In [85]:
def evaluate_model(y_true, y_pred, name):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred)
    }

# 9.2 Collect and Display Results

In [86]:
results = []
for name, y_pred in predictions.items():
    results.append(evaluate_model(y_test, y_pred, name))
results_df = pd.DataFrame(results)
results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.772727,0.692308,0.654545,0.672897
1,Decision Tree,0.688312,0.550725,0.690909,0.612903
2,Random Forest,0.766234,0.666667,0.690909,0.678571
3,SVM,0.753247,0.666667,0.618182,0.641509


# 9.3 Bar Chart Comparison

In [87]:
fig = px.bar(results_df, x='Model', y=['Accuracy','Precision','Recall','F1 Score'],
             title=' Model Performance', barmode='group')
fig.show()

# 10. Confusion Matrix (Random Forest)

In [88]:
cm = confusion_matrix(y_test, predictions['Random Forest'])
fig = px.imshow(cm, text_auto=True, title=' Confusion Matrix – Random Forest',
                labels=dict(x='Predicted', y='Actual'))
fig.show()

# 11. ROC Curves

In [89]:
fig = go.Figure()
for name, prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_auc = auc(fpr, tpr)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'{name} (AUC={roc_auc:.2f})'))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Random', line=dict(dash='dash')))
fig.update_layout(title='ROC Curves', xaxis_title='False Positive Rate', yaxis_title='True Positive Rate')
fig.show()

# 12. Feature Importance (Random Forest)

In [90]:
importances = models['Random Forest'].feature_importances_
feat_imp = pd.DataFrame({'feature': X.columns, 'importance': importances})
feat_imp = feat_imp.sort_values('importance', ascending=False).head(10)
fig = px.bar(feat_imp, x='importance', y='feature', orientation='h',
             title=' Top 10 Feature Importances')
fig.show()

# 13. Direct Prediction on New Data

# Now you can input new patient data and get an instant prediction from the best model (Random Forest).
# Enter values for each feature when prompted.

In [95]:
# Load the best model (Random Forest) and scaler
best_model = models['Random Forest']

def predict_new():
    print("Enter patient details for diabetes prediction:\n")
    pregnancies = float(input("Pregnancies: "))
    glucose = float(input("Glucose: "))
    blood_pressure = float(input("BloodPressure: "))
    skin_thickness = float(input("SkinThickness: "))
    insulin = float(input("Insulin: "))
    bmi = float(input("BMI: "))
    dpf = float(input("DiabetesPedigreeFunction: "))
    age = float(input("Age: "))

    # Create a DataFrame with the same columns as training
    input_data = pd.DataFrame([[pregnancies, glucose, blood_pressure, skin_thickness,
                                 insulin, bmi, dpf, age]],
                               columns=['Pregnancies','Glucose','BloodPressure','SkinThickness',
                                        'Insulin','BMI','DiabetesPedigreeFunction','Age'])

    # Add engineered features (must match training)
    input_data['Glucose_BMI'] = input_data['Glucose'] * input_data['BMI']
    # AgeGroup dummies: we need to replicate the encoding from training
    # For simplicity, we'll use the same bins and assign the correct dummy
    bins = [20,30,40,50,60,100]
    labels = ['20-30','30-40','40-50','50-60','60+']
    age_val = input_data['Age'].iloc[0]
    age_group = pd.cut([age_val], bins=bins, labels=labels)[0]
    # Create dummy columns (all False initially)
    for group in labels[1:]:  # skip first as reference
        input_data[f'AgeGroup_{group}'] = 1 if age_group == group else 0

    # Ensure column order matches training
    input_data = input_data[X.columns]  # X columns from training

    # Scale
    input_scaled = scaler.transform(input_data)

    # Predict
    pred = best_model.predict(input_scaled)[0]
    prob = best_model.predict_proba(input_scaled)[0][1]

    print("\n Prediction Result:")
    if pred == 1:
        print(f"The model predicts **DIABETES** with probability {prob:.2f}")
    else:
        print(f"The model predicts **NO DIABETES** with probability {1-prob:.2f}")

# Run the prediction function
predict_new()

Enter patient details for diabetes prediction:

Pregnancies: 1
Glucose: 160
BloodPressure: 90
SkinThickness: 120
Insulin: 100
BMI: 45
DiabetesPedigreeFunction: 1
Age: 60

 Prediction Result:
The model predicts **DIABETES** with probability 0.77


# 14. Save Model and Preprocessing Objects

In [92]:
import joblib
joblib.dump(best_model, 'diabetes_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(X.columns.tolist(), 'feature_names.pkl')
print("Model, scaler, and feature names saved.")

Model, scaler, and feature names saved.


# 15. Conclusion

In [93]:
print("Diabetes prediction project completed successfully!")
print(" Output files: diabetes_model.pkl, scaler.pkl, feature_names.pkl")
print(" You can now use the saved model to make predictions on new data.")

Diabetes prediction project completed successfully!
 Output files: diabetes_model.pkl, scaler.pkl, feature_names.pkl
 You can now use the saved model to make predictions on new data.
